```
<08_four_kinds.ipynb>

제미나이 의존도: 70-80%

신선한 사과, 상한 사과, 신선한 바나나, 상한 바나나 데이터셋을 roboflow에서 구해왔다.
나는 사과도 바나나도 하나의 데이터셋에 있는 걸 원했는데, 역시.. 데이터셋은 그렇게 친절하지 않았다.
그래서 roboflow에서 4개 각각의 데이터셋을 export해서 통합 데이터셋을 만든 다음, 몰아넣었다.

하면서, 변수 네이밍이 참 중요하다고 느꼈다. path 같은 짧고 좋지만, 무슨 경로인지 나 말고 다른 사람은 알기 힘들다.
변수명이 조금 길어지더라도 직관적이게 누구나 이해할 수 있도록 네이밍하는 게 좋다.
그래서 처음 코드를 작성할 때 시간이 좀 소요되더라도 번역기를 사용하든지 해서 직관적이고 이해할 수 있는 변수명으로 네이밍 하자.
```

fresh_apple - train(120)
rotten_apple - train(1398) / valid(399) / test(203)
fresh_banana - train(420) / valid(120) / test(61)
rotten_banana - train(739) / valid(211) / test(105)

fresh_apple을 제외하고, 각각 train 폴더에 몰빵한다.

In [33]:
import os
import shutil
from pathlib import Path
import random
from ultralytics import YOLO
import time

In [12]:
path = "dataset/rotten_banana/test/images"

list_ = os.listdir(path)
print(len(list_))

105


In [6]:
# valid to train, test to train
def move_to_train(fruits):
    common_path = f"dataset/{fruits}"
    train_path = os.path.join(common_path, "train")
    train_images_path = os.path.join(train_path, "images")
    train_labels_path = os.path.join(train_path, "labels")
    
    valid_path = os.path.join(common_path, "valid")
    valid_images_path = os.path.join(valid_path, "images")
    valid_labels_path = os.path.join(valid_path, "labels")
    
    test_path = os.path.join(common_path, "test")
    test_images_path = os.path.join(test_path, "images")
    test_labels_path = os.path.join(test_path, "labels")

    val_to_train_count = 0
    test_to_train_count = 0

    # valid to train
    for valid_image in os.listdir(valid_images_path):
        pure_name = Path(valid_image).stem
        txt_name = f"{pure_name}.txt"
        if txt_name in os.listdir(valid_labels_path): # 이미지, 텍스트파일 둘 다 존재
            full_image_path = os.path.join(valid_images_path, valid_image)
            full_txt_path = os.path.join(valid_labels_path, txt_name)
            
            shutil.move(full_image_path, train_images_path)
            shutil.move(full_txt_path, train_labels_path)
            val_to_train_count += 1

    # test to train
    for test_image in os.listdir(test_images_path):
        pure_name = Path(test_image).stem
        txt_name = f"{pure_name}.txt"
        if txt_name in os.listdir(test_labels_path): # 이미지, 텍스트파일 둘 다 존재
            full_image_path = os.path.join(test_images_path, test_image)
            full_txt_path = os.path.join(test_labels_path, txt_name)
            
            shutil.move(full_image_path, train_images_path)
            shutil.move(full_txt_path, train_labels_path)
            test_to_train_count += 1

    print(f"{val_to_train_count} + {test_to_train_count} = {val_to_train_count + test_to_train_count}")

In [33]:
# move_to_train("rotten_apple")
move_to_train("fresh_banana")
move_to_train("rotten_banana")

120 + 60 = 180
211 + 105 = 316


파일명 앞에 
fresh_apple은 0
rotten_apple은 1
fresh_banana은 2
rotten_banana은 3
붙이기..


In [12]:
def change_file_name(class_id, fruit):
    train_path = f"dataset/{fruit}/train"
    images_path = os.path.join(train_path, "images")
    labels_path = os.path.join(train_path, "labels")

    count = 0

    # 개선 1: labels 디렉토리의 파일 목록을 미리 set으로 받아 탐색 성능 향상
    label_set = set(os.listdir(labels_path))
    
    for i, image in enumerate(os.listdir(images_path)):
        # class_id = 0
        # fruit = "fresh_apple"
        pure_name, ext = os.path.splitext(image)
        # pure_name = Path(image).stem
        # print(pure_name) # abc
        # print(ext) # .jpg
        txt_name = f"{pure_name}.txt"
        if txt_name in label_set: # 이미지와 텍스트 둘 다 존재한다면!
            full_image_path = os.path.join(images_path, image)
            full_txt_path = os.path.join(labels_path, txt_name)
            
            # print(full_image_path) # dataset/../abc.jpg
            # print(full_txt_path) # dataset/../abc.txt
            # print(txt_name) # abc.txt
            numbering = f"{i+1:04d}"
            
            # print(number) # 0001
    
            new_name = f"{class_id}_{fruit}_{numbering}"
            # print(new_name) # 0_fresh_apple_0001
            new_image_name = f"{new_name}{ext}"
            new_txt_name = f"{new_name}.txt"
            # print(new_image_name, new_txt_name) # 0_fresh_apple_0001.png 0_fresh_apple_0001.txt
    
            new_image_path = os.path.join(images_path, new_image_name)
            new_txt_path = os.path.join(labels_path, new_txt_name)
            # print(new_image_path, new_txt_path)

            # 개선 2: .txt 파일 내부의 클래스 ID를 새로운 class_id로 수정하는 로직 추가
            with open(full_txt_path, 'r') as f:
                lines = f.readlines()

            modified_lines = []
            for line in lines:
                parts = line.split()
                if parts:
                    parts[0] = str(class_id) # 맨 앞 번호를 매개변수 class_id로 강제 치환
                    modified_lines.append(" ".join(parts) + "\n")


            with open(full_txt_path, "w") as f:
                f.writelines(modified_lines)
            
            # 파일명 변경
            os.rename(full_image_path, new_image_path)
            os.rename(full_txt_path, new_txt_path)

            count += 1
            # break
    print(f"{count}개 닉변 완료")

In [13]:
change_file_name(0, "fresh_apple")

200개 닉변 완료


In [14]:
change_file_name(1, "rotten_apple")
change_file_name(2, "fresh_banana")
change_file_name(3, "rotten_banana")

981개 닉변 완료
301개 닉변 완료
504개 닉변 완료


In [24]:
change_file_name(1, "rotten_apple")

1398개 닉변 완료


In [15]:
random.seed(42)
path = "dataset/fresh_apple/train"
fresh_apple_image_path = os.path.join(path, "images")
fresh_apple_txt_path = os.path.join(path, "labels")

fresh_apple_image_list = os.listdir(fresh_apple_image_path)
print(len(fresh_apple_image_list))

random.shuffle(fresh_apple_image_list)

200


In [9]:
move_to_train("fresh_apple2")

20 + 20 = 40


In [27]:
def length(fruit):
    list_ = ["train", "valid", "test"]
    for item in list_:
        
        path = f"dataset/{fruit}/{item}"
        if not os.path.exists(path): continue
        
        image_path = os.path.join(path, "images")
        txt_path = os.path.join(path, "labels")
        
        images = [f for f in os.listdir(image_path) if f.endswith(('.jpg', '.jpeg', '.png', '.webp'))]
        txts = [f for f in os.listdir(txt_path) if f.endswith(('.txt'))]
    
        print(f"{fruit} {item} 이미지 개수: {len(images)}, 텍스트 개수: {len(txts)}")
    print()

In [28]:
length("fresh_apple")
length("rotten_apple")
length("fresh_banana")
length("rotten_banana")

fresh_apple train 이미지 개수: 200, 텍스트 개수: 200
fresh_apple valid 이미지 개수: 0, 텍스트 개수: 0
fresh_apple test 이미지 개수: 0, 텍스트 개수: 0

rotten_apple train 이미지 개수: 1398, 텍스트 개수: 1398

fresh_banana train 이미지 개수: 301, 텍스트 개수: 301
fresh_banana valid 이미지 개수: 0, 텍스트 개수: 0
fresh_banana test 이미지 개수: 0, 텍스트 개수: 0

rotten_banana train 이미지 개수: 504, 텍스트 개수: 504
rotten_banana valid 이미지 개수: 0, 텍스트 개수: 0
rotten_banana test 이미지 개수: 0, 텍스트 개수: 0



In [9]:
def split_and_integrated_dataset(fruit):
    # fruit = "fresh_apple"
    # 출발지 경로 (Source)
    src_base_path = f"dataset/{fruit}/train"
    print(f"src_base_path: {src_base_path}")
    src_image_dir = os.path.join(src_base_path, "images")
    src_txt_dir = os.path.join(src_base_path, "labels")
    
    # 목적지 경로 (Destination)
    dest_train_path = "dataset/integrated_dataset/train"
    dest_train_image_path = os.path.join(dest_train_path, "images")
    dest_train_txt_path = os.path.join(dest_train_path, "labels")
    
    dest_valid_path = "dataset/integrated_dataset/valid"
    dest_valid_image_path = os.path.join(dest_valid_path, "images")
    dest_valid_txt_path = os.path.join(dest_valid_path, "labels")
    
    dest_test_path = "dataset/integrated_dataset/test"
    dest_test_image_path = os.path.join(dest_test_path, "images")
    dest_test_txt_path = os.path.join(dest_test_path, "labels")
    
    images = [f for f in os.listdir(src_image_dir) if f.endswith(('.jpg', '.jpeg', '.png', '.webp'))]
    txts = [f for f in os.listdir(src_txt_dir) if f.endswith(('.txt'))]
    
    target_count = 504
    if len(images) > target_count:
        selected_images = random.sample(images, target_count)
    else:
        selected_images = images
    
    
    train_size = int(len(selected_images) * 0.8)
    val_size = int(len(selected_images) * 0.1)
    
    train_list = selected_images[:train_size]
    val_list = selected_images[train_size:train_size + val_size]
    test_list = selected_images[train_size + val_size:]
    
    
    # train_list는 train으로, val_list는 valid로, test_list는 test로
    
    # 분할된 리스트와 각각의 목적지 경로를 매핑한 딕셔너리 만들기
    split_maps = {
        "train": {"list": train_list, "img_dir": dest_train_image_path, "txt_dir": dest_train_txt_path}, 
        "valid": {"list": val_list, "img_dir": dest_valid_image_path, "txt_dir": dest_valid_txt_path}, 
        "test": {"list": test_list, "img_dir": dest_test_image_path, "txt_dir": dest_test_txt_path}
    }
    
    # txt 존재 여부 검사를 위해 set 미리 만들어두기 (속도 최적화)
    txt_set = set(txts)
    
    # 딕셔너리를 돌면서 단 하나의 for문 구조로 모두 해결하기
    for split_name, info in split_maps.items():
        count = 0
        current_list = info["list"]
        dest_img_dir = info["img_dir"]
        dest_txt_dir = info["txt_dir"]
    
        for image in current_list:
            pure_name, ext = os.path.splitext(image)
            txt_file_name = f"{pure_name}.txt"
    
            if txt_file_name in txt_set:
                full_image_path = os.path.join(src_image_dir, image)
                dest_full_image_path = os.path.join(dest_img_dir, image)
    
                full_txt_path = os.path.join(src_txt_dir, txt_file_name)
                dest_full_txt_path = os.path.join(dest_txt_dir, txt_file_name)
    
                # print(f"split_name: {split_name}, size: {len(current_list)}")
                # print(f"full_image_path: {full_image_path}, dest_full_image_path: {dest_full_image_path}")
                # print(f"full_txt_path: {full_txt_path}, dest_full_txt_path: {dest_full_txt_path}")
                # break
                shutil.copy(full_image_path, dest_full_image_path)
                shutil.copy(full_txt_path, dest_full_txt_path)
                count += 1
    
        print(f"{count}개 integrated_dataset/{split_name}으로 이동 완료")

In [10]:
split_and_integrated_dataset("fresh_apple")

src_base_path: dataset/fresh_apple/train
160개 integrated_dataset/train으로 이동 완료
20개 integrated_dataset/valid으로 이동 완료
20개 integrated_dataset/test으로 이동 완료


In [13]:
split_and_integrated_dataset("rotten_apple")
split_and_integrated_dataset("fresh_banana")
split_and_integrated_dataset("rotten_banana")

src_base_path: dataset/rotten_apple/train
403개 integrated_dataset/train으로 이동 완료
50개 integrated_dataset/valid으로 이동 완료
51개 integrated_dataset/test으로 이동 완료
src_base_path: dataset/fresh_banana/train
240개 integrated_dataset/train으로 이동 완료
30개 integrated_dataset/valid으로 이동 완료
31개 integrated_dataset/test으로 이동 완료
src_base_path: dataset/rotten_banana/train
403개 integrated_dataset/train으로 이동 완료
50개 integrated_dataset/valid으로 이동 완료
51개 integrated_dataset/test으로 이동 완료


In [29]:
length("integrated_dataset")

integrated_dataset train 이미지 개수: 1206, 텍스트 개수: 1206
integrated_dataset valid 이미지 개수: 150, 텍스트 개수: 150
integrated_dataset test 이미지 개수: 153, 텍스트 개수: 153



In [36]:
# data.yaml 파일 작성하기.
yaml_file_path = "dataset/integrated_dataset/data.yaml"

yaml_content = """train: ../train/images
val: ../valid/images
test: ../test/images

nc: 4
names: ['fresh_apple', 'rotten_apple', 'fresh_banana', 'rotten_banana']
"""

with open(yaml_file_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)
    

In [37]:
current_dir = os.getcwd()

model = YOLO("yolov8n.pt")

start_time = time.time()

results = model.train(
    data="dataset/integrated_dataset/data.yaml", 
    epochs=30, 
    imgsz=640, 
    device="mps", 
    project=os.path.join(current_dir, "runs"), 
    name="four_kinds"
)

end_time = time.time()

elapsed_time = int(end_time - start_time)
mins, secs = divmod(elapsed_time, 60)
print(f"Time Taken: {mins}:{secs}")

New https://pypi.org/project/ultralytics/8.4.102 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.87 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/integrated_dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=four_kin